---
title: Dataset and DataLoader
numbering: false
---

# 0. 개요
+ Dataset: i번째 샘플 하나를 어떻게 꺼내는가
+ DataLoader: 샘플 여러 개를 묶고 섞고 반복하는가

# 1. FashionMNIST

In [1]:
import torch
from torchvision import datasets
from torchvision.transforms import ToTensor

train_ds = datasets.FashionMNIST(root="data", train=True, download=True, transform=ToTensor())
test_ds = datasets.FashionMNIST(root="data", train=False, download=True, transform=ToTensor())

x, y = train_ds[0] # Dataset 은 인덱싱할 경우 샘플 하나를 준다

In [2]:
print(x.shape)

torch.Size([1, 28, 28])


In [3]:
print(type(y))

<class 'int'>


In [4]:
train_ds.classes # 클래스의 종류 출력

['T-shirt/top',
 'Trouser',
 'Pullover',
 'Dress',
 'Coat',
 'Sandal',
 'Shirt',
 'Sneaker',
 'Bag',
 'Ankle boot']

# 2. Transform
+ 원본 데이터를 모델이 입력받을 수 있는 형태로 바꾼다
+ transform 을 하지 않으면 PIL(파이썬 이미지 객체)로 나오게 된다. 모델은 텐서만 받으니 변환이 필요하다. 따라서 ToTensor()는 한 번에 세 가지 일을 하게 된다
    + 모양: (채널, 높이, 너비)로 바꿈 > (B, C, H, W)
    + 자료형: float32 로 바꿈
    + 값 범위: 0~1로 바꿈

In [5]:
# 여러 변환을 한 번에 이어 붙이는 경우 Compose() 를 사용

from torchvision.transforms import Compose, Normalize
tf = Compose([ToTensor(), Normalize((0.5,), (0.5,))]) # (x-0.5)/(0.5) > 범위가 -1 ~ 1 이 된다.

# 3. DataLoader

In [6]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
xb, yb = next(iter(train_loader)) # 첫 번째 배치만 하나 꺼내보기+

+ 묶기 : 샘플 64개의 이미지랄 쌓아 텐서하나로, 라벨 64개도 텐서 하나로 만들어준다.
+ 섞기 : 학습용은 shuffle=True, 검증 및 테스트 용은 shuffle=False 로 평가는 순서가 상관없고 재현가능해야됨
+ drop_last=True: 배치로 개수를 나눴을 때 딱 떨어지지 않으면 남은 자투리 데이터를 버린다
+ 재현성: 섞는 순서를 고정하려면 manual_seed처럼 생성기를 넘겨야 함

In [7]:
# DataLoader(train_ds, batch_size=64, shuffle=True, generator=torch.Generator.manual_seed(42))

+ `num_workers` : 데이터를 여러 프로세스로 병렬로 로딩하는 옵션

# 4. 검증 세트 나누기

In [8]:
from torch.utils.data import random_split
train_set, val_set = random_split(train_ds, [50000, 10000], generator=torch.Generator().manual_seed(0))

+ 테스트세트는 학습 중에는 절대로 건드리지 않는다.
+ 하이퍼 파라미터를 고르거나 어느 체크포인트가 좋은지 판단하는 것은 validation set 으로 진행한다
+ 테스트 점수를 보고 모델을 고르면 처음 보는 데이터에서의 성능이 아니다.

# 5. 나만의 데이터셋을 만드는 방법

In [9]:
from torch.utils.data import Dataset

class SqaureDataset(Dataset):
    def __init__(self, n): # 무거운 준비는 여기서 한번에 처리한다
        self.x = torch.arange(n, dtype=torch.float32)

    def __len__(self): # 샘플의 개수
        return len(self.x)

    def __getitem__(self, i):
        return self.x[i], self.x[i] ** 2

sq = SqaureDataset(10)

In [10]:
len(sq) # 10

10

In [11]:
sq[3] # tensor(3.), tensor(9.)

(tensor(3.), tensor(9.))

In [12]:
next(iter(DataLoader(sq, batch_size=4))) # [tensor([0., 1., 2., 3.]), tensor([0., 1., 4., 9.])]

[tensor([0., 1., 2., 3.]), tensor([0., 1., 4., 9.])]